<a href="https://colab.research.google.com/github/Megeeee/AgenticAI/blob/AntrophicAgent/CNN_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim

In [ ]:
class CNNModel(nn.Module):
  def __init__(self,height,width):
    super().__init__()
    self.model = nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,padding=1), #32x28x28, 32xhxw
        nn.ReLU(),
        nn.MaxPool2d(2,2),#32x14x14 , 32xhxw/4

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),#64x7x7, 64xhxw/16

        nn.Flatten(),
        nn.Linear(height*width*4,100),
        nn.ReLU(),
        nn.Linear(100,10)
    )

  def forward(self,x):
    return self.model(x)

In [ ]:
def train(model,trainloader,epochsize = 10,lr = 0.002, loss = nn.CrossEntropyLoss):
  loss = loss()

  optimizer = torch.optim.Adam(model.parameters(), lr=lr)
  print("Training on:}",torch.cuda.get_device_name(0))

  lossL = []
  #train
  for epoch in range(epochsize):
    running_loss = 0.0
    for i,data in enumerate(trainloader,0):
      inputs,labels = data
      input,labels = inputs.to("cuda"),labels.to("cuda")

      optimizer.zero_grad()

      outputs = model(input)
      l = loss(outputs,labels)
      running_loss += l.item()
      l.backward()
      optimizer.step()
      if i % 200 == 199:
              print(f'[Epoch: {epoch + 1}, Batch: {i + 1}] loss: {running_loss / 200:.3f}')
              lossL.append(running_loss/200)
              running_loss = 0.0


  plt.plot(lossL)
  plt.show()
  print('Finished Training')

In [ ]:
def test(model,testloader):
  #test
  model.eval()
  correct=0
  total= 0

  with torch.no_grad():
    for data in testloader:
      inputs,labels = data
      inputs,labels = inputs.to("cuda"),labels.to("cuda")
      outputs = model(inputs)

      correct += torch.sum(torch.max(outputs,1)[1]==labels).item()
      total += len(labels)

  print("Accuracy:",correct/total)
  print("Correct:",correct)
  print("total:",total)

In [ ]:
def MNISTtrainloaderload():
  transform = transforms.Compose([transforms.ToTensor(),
                                  transforms.RandomRotation(degrees=25,fill = 0),
                                  transforms.Normalize((0.5,), (0.5,))])
  trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
  trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
  return trainloader


In [ ]:
def MNISTtestloaderload():
  transform = transforms.Compose([transforms.ToTensor(),
                                 transforms.RandomRotation(degrees=25,fill = 0),
                                transforms.Normalize((0.5,), (0.5,))])
  testset =  datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)
  testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

  return testloader
